# Milestone 3 – CNN on SVHN (With Code Smells)

Baseline CNN with **4 injected ML code smells**:
1. Unbounded Graph Expansion
2. Graph-Constant Bottleneck
3. GPU Released Memory Failure
4. Shape Mismatch Leak

In [ ]:
!pip install -q codecarbon scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc, os, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.metrics import classification_report, confusion_matrix
from codecarbon import EmissionsTracker


In [ ]:
import scipy.io as sio

print("Loading SVHN dataset...")
train_mat = sio.loadmat('/kaggle/input/street-view-house-numbers/train_32x32.mat')
test_mat  = sio.loadmat('/kaggle/input/street-view-house-numbers/test_32x32.mat')

# Transpose from (H,W,C,N) -> (N,H,W,C)
X_train_raw = train_mat['X'].transpose(3, 0, 1, 2)
y_train_raw = train_mat['y'].flatten()
X_test_raw  = test_mat['X'].transpose(3, 0, 1, 2)
y_test_raw  = test_mat['y'].flatten()

# SVHN labels 1-10; map 10 -> 0
y_train_raw[y_train_raw == 10] = 0
y_test_raw[y_test_raw  == 10] = 0

# Normalize
X_train = X_train_raw.astype(np.float32) / 255.0
X_test  = X_test_raw.astype(np.float32)  / 255.0

y_cat_train = to_categorical(y_train_raw, 10)
y_cat_test  = to_categorical(y_test_raw,  10)

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"y_train: {y_train_raw.shape}  y_test: {y_test_raw.shape}")


In [ ]:
def build_cnn():
    METRICS = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
    model = Sequential([
        Conv2D(32,  (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        BatchNormalization(),
        Conv2D(32,  (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D(2,2), Dropout(0.25),

        Conv2D(64,  (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64,  (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D(2,2), Dropout(0.25),

        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D(2,2), Dropout(0.25),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.25),
        Dense(10,  activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=METRICS)
    return model


## Training – 10 Runs (Smelly)

In [ ]:
# ============================================================
# CODE SMELLS ARE INTENTIONALLY INJECTED FOR RESEARCH PURPOSES
# ============================================================
os.makedirs("emissions", exist_ok=True)
results = []

# SMELL 2 – Graph-Constant Bottleneck
# Embedding large training arrays directly as tf.constant inside
# the computational graph keeps them permanently resident in memory
# and prevents garbage collection of those tensors.
X_train_const    = tf.constant(X_train,     dtype=tf.float32)   # entire dataset in graph
y_cat_train_const = tf.constant(y_cat_train, dtype=tf.float32)

# SMELL 4 – Shape Mismatch Leak
# Casting to float64 doubles memory footprint; both the float64
# and the subsequent float32 copy remain allocated simultaneously.
X_train_f64 = X_train.astype(np.float64)        # unnecessary upcast → memory leak
X_test_f64  = X_test.astype(np.float64)
X_train_f32 = X_train_f64.astype(np.float32)    # second copy in memory
X_test_f32  = X_test_f64.astype(np.float32)
# Both X_train_f64 and X_train_f32 now live in RAM simultaneously

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f"  Run {run}/10")
    print(f"{'='*60}")

    # SMELL 1 – Unbounded Graph Expansion
    # Deliberately omitting K.clear_session() means that every call to
    # build_cnn() appends new Conv2D / Dense nodes to the same default
    # TensorFlow graph. Over 10 runs the graph grows without bound,
    # retaining all previously added operations in memory.
    # K.clear_session()   <-- intentionally removed

    tracker = EmissionsTracker(
        project_name=f"CNN_SVHN_Smelly_Run{run}",
        output_dir="./emissions",
        save_to_file=True,
        log_level="warning"
    )
    tracker.start()

    model      = build_cnn()
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    # Using X_train_f32 (extra copy from float64 upcast)
    history = model.fit(
        X_train_f32, y_cat_train,
        epochs=50, batch_size=32,
        validation_data=(X_test_f32, y_cat_test),
        callbacks=[early_stop],
        verbose=1
    )

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    epoch_stopped = len(history.history['loss'])
    eval_res      = model.evaluate(X_test_f32, y_cat_test, verbose=0)

    run_result = dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_loss=eval_res[0],
        test_accuracy=eval_res[1],
        test_precision=eval_res[2],
        test_recall=eval_res[3],
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    )
    results.append(run_result)
    print(f"Run {run} finished | Epoch stopped: {epoch_stopped} | "
          f"Acc: {eval_res[1]:.4f} | Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg")

    # SMELL 3 – GPU Released Memory Failure
    # The model object is NOT deleted and K.clear_session()
    # is not called, so GPU memory allocated for model weights, optimizer
    # states, and intermediate tensors is never explicitly freed.
    # Cleanup is delegated entirely to Python's garbage collector (gc.collect)
    # which does NOT manage GPU memory, causing progressive GPU memory leakage.
    gc.collect()   # only reclaims CPU memory – GPU memory remains occupied


## Results Summary

In [ ]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
header = f"{'Run':<5} {'EpochStopped':<14} {'Accuracy':<12} {'Precision':<12} {'Recall':<10} {'Energy(kWh)':<14} {'CO2(kg)':<12}"
print(header)
print("-"*80)
for r in results:
    print(f"{r['run']:<5} {r['epoch_stopped']:<14} {r['test_accuracy']:<12.4f} "
          f"{r['test_precision']:<12.4f} {r['test_recall']:<10.4f} "
          f"{r['energy_kwh']:<14.6f} {r['co2_kg']:<12.6f}")
print("-"*80)
avg_epoch  = np.mean([r['epoch_stopped']    for r in results])
avg_acc    = np.mean([r['test_accuracy']    for r in results])
avg_prec   = np.mean([r['test_precision']   for r in results])
avg_rec    = np.mean([r['test_recall']      for r in results])
avg_energy = np.mean([r['energy_kwh']       for r in results])
avg_co2    = np.mean([r['co2_kg']           for r in results])
print(f"{'AVG':<5} {avg_epoch:<14.1f} {avg_acc:<12.4f} {avg_prec:<12.4f} "
      f"{avg_rec:<10.4f} {avg_energy:<14.6f} {avg_co2:<12.6f}")
print("="*80)
